# Create Doris Duke Foundation Awards (MULTI-SCHEME, RSC stream + Sanity GROQ)

Doris Duke Charitable Foundation (US) funds organizations directly across Environment / Arts / Child Well-being / Medical Research / Building Bridges, AND runs the **Doris Duke Artist Awards** (DDAA) — an individual-recipient fellowship for performing artists active since 2012.

**Awarding body:** Doris Duke Charitable Foundation - F4320306134 (US, DOI 10.13039/100001025)

**Sources (two on the same pull):**
1. **Organizational grants** — embedded as JSON inside the React Server Components stream on `/grantees` (Next.js). Each card carries Date, Organization Name, Program, Amount (`$$X,YYY`), Description, Duration. ~128 unique grants 2021-2024.
2. **DDAA artists + individual grantees** — exposed via the site's Sanity CMS public GROQ API at `w64oxp25.apicdn.sanity.io`. `*[_type=="artist"]` returns 153 artist documents with name, slug, subtitle, and references to `artistEdition` (year) and `granteeDiscipline`. `*[_type=="grantee"]` returns 7 non-artist individual grantees.

**First Sanity GROQ ingest on the project.** Discovery path: Next.js with RSC streaming, RSC payload had no clean JSON dump, BUT cdn.sanity.io image refs exposed the projectId (`w64oxp25`) and dataset (`production`). The unauthenticated GROQ endpoint then serves all published docs with full schema resolution.

**Schema choices:**
- Three `funder_scheme` values within the same provenance:
  - `Doris Duke Foundation Grant` (128 rows) — org-level, amount populated, funding_type=`grant`
  - `Doris Duke Artist Awards` (153 rows) — individual artist, amount NULL (§6.7-waived), funding_type=`fellowship`
  - `Doris Duke Grantee` (7 rows) — other individual fellowships, amount NULL (§6.7-waived), funding_type=`fellowship`
- `funder_award_id` = `doris-duke-{kind}-{slug}` (kind ∈ grant / artist / grantee) so the 3 schemes never collide.
- `lead_investigator` shape adapts to source_kind:
  - Organizational grants: org-only (given/family/orcid NULL, affiliation.name = grantee org, country=US)
  - Artists / grantees: individual (given/family populated, affiliation.name NULL — source doesn't expose institutional affiliation, country=US)

**Amount and currency:**
- Org grants: USD parsed from the source's `Amount` field (the RSC payload encodes it as `$$X,YYY` — the double dollar is a Sanity CMS portable-text quirk; we strip both via regex).
- DDAA artists + individual grantees: amount NULL with **§6.7-waiver**. The DDF Artist Awards amount has changed over its 14-year history and the live site does NOT publish a verbatim per-year/per-recipient figure in machine-extractable form. Boss can backfill via SQL CASE on edition year if the published historical schedule is acceptable (rough public knowledge: ~$275K through 2023, $325K from 2024+).

**Coverage scope:** the foundation's grant database publicly exposes only recent grants (2021-2024 for orgs) plus the DDAA artist roster (2012-2026). DDF's full historical record (~4,330 publications cited in OpenAlex) is presumably much larger; pre-2021 organizational grants aren't reachable from the public site.

**S3 location:** `s3a://openalex-ingest/awards/doris_duke/doris_duke.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.doris_duke_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/doris_duke/doris_duke.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.doris_duke_raw;

## Step 1.5: Inspect raw + per-scheme distribution

Expected: ~288 rows total (~128 org grants + 153 artists + 7 grantees). Amount populated only on org grants (44% overall). Year range 2012-2026.

In [ ]:
%sql
DESCRIBE openalex.awards.doris_duke_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.doris_duke_raw LIMIT 5;

In [ ]:
%sql
-- Source-kind / scheme distribution.
SELECT source_kind, scheme, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       COUNT(year) AS has_year
FROM openalex.awards.doris_duke_raw
GROUP BY source_kind, scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Year range (org grants 2021-2024; DDAA artists 2012-2026).
SELECT scheme, MIN(year) AS min_year, MAX(year) AS max_year, COUNT(DISTINCT year) AS years
FROM openalex.awards.doris_duke_raw
WHERE year IS NOT NULL
GROUP BY scheme;

In [ ]:
%sql
-- DDAA artists by edition + discipline.
SELECT year, discipline, COUNT(*) AS rows
FROM openalex.awards.doris_duke_raw
WHERE scheme = 'Doris Duke Artist Awards'
GROUP BY year, discipline
ORDER BY year DESC, rows DESC
LIMIT 20;

## Step 1.6: Fail-fast - verify Doris Duke funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306134;

## Step 2: Transform to award schema

`lead_investigator` shape adapts per source_kind: organizational grants get org-only (grantee_org as affiliation.name, given/family NULL); artists and individual grantees get individual (given/family populated, affiliation.name NULL). `funding_type` derived per scheme. Country=US throughout (Doris Duke is US-only and funds US-based recipients).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.doris_duke_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306134  -- Doris Duke Charitable Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CASE
        WHEN n.source_kind = 'organizational_grant'
            THEN CONCAT('Doris Duke Grant - ', n.grantee_org,
                        CASE WHEN n.year IS NOT NULL THEN CONCAT(' (', n.year, ')') ELSE '' END)
        ELSE CONCAT('Doris Duke ', n.scheme, ' - ', n.recipient_name,
                    CASE WHEN n.year IS NOT NULL THEN CONCAT(' (', n.year, ')') ELSE '' END)
    END AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    n.funding_type,
    n.scheme AS funder_scheme,
    'doris_duke' AS provenance,
    TRY_TO_DATE(n.approved_on, 'yyyy-MM-dd') AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.source_kind = 'organizational_grant' AND n.grantee_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.approved_on, 'yyyy-MM-dd') AS role_start,
            struct(
                n.grantee_org AS name,
                CAST('US' AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        WHEN n.recipient_name IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,  -- source doesn't expose institutional affiliation for artists/grantees
                CAST('US' AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.doris_duke_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND (n.grantee_org IS NOT NULL OR n.recipient_name IS NOT NULL);

## Step 3: Insert into openalex_awards_raw at priority 143

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'doris_duke' AND priority = 143;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    143 AS priority  -- Doris Duke Charitable Foundation
FROM openalex.awards.doris_duke_awards;

## Step 6: Verification

§6.7 amount-coverage is **partial**: ~44% (orgs populated, DDAA artists + individual grantees §6.7-waived). Year coverage ~98%. Year range 2012-2026.

In [ ]:
%sql
SELECT COUNT(*) AS total_doris_duke_rows FROM openalex.awards.doris_duke_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.doris_duke_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_amount expected ~44% (orgs only; artists/grantees §6.7-waived).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_recipient,
    COUNT(start_year) AS has_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.doris_duke_awards;

In [ ]:
%sql
-- §6.7 amount split by scheme. Orgs must be 100% (USD); artists + grantees 0% (§6.7-waived).
SELECT funder_scheme,
       COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
       collect_set(currency) AS currencies
FROM openalex.awards.doris_duke_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- funding_type split.
SELECT funding_type, COUNT(*) AS rows
FROM openalex.awards.doris_duke_awards
GROUP BY funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.doris_duke_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA — mix of org grants + artists.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme, funding_type, start_year, amount,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS grantee_org
FROM openalex.awards.doris_duke_awards
ORDER BY start_year DESC NULLS LAST, funder_scheme
LIMIT 10;